# 5. Conclusiones Globales del Proyecto

**Materia:** Aprendizaje Automático (Machine Learning)  
**Trabajo Práctico 2:** Clasificación Supervisada  
**Dataset:** Bank Marketing (UCI / Kaggle)  
**Objetivo:** Integrar los hallazgos y responder formalmente a las cuatro preguntas fundamentales del trabajo práctico:
1. ¿Qué modelos funcionaron mejor y por qué? (relación con el sesgo y la varianza).
2. ¿Los datos satisfacen los supuestos teóricos de cada modelo?
3. ¿Qué clasificación de errores es más relevante para el problema de negocio?
4. Limitaciones del sistema y propuestas de mejora.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path("../data")
FIGURES_PATH = Path("../figures")

## 5.1 ¿Qué modelos funcionaron mejor y por qué? (Compromiso Sesgo-Varianza)

### Ranking de Desempeño (ROC-AUC de validación, modelos ajustados):
1. **Random Forest ($\text{ROC-AUC} \approx 0.798$, `max_depth = 8`, set 1):**
   - *Fundamento:* Al ser un ensamble de árboles con profundidad limitada, combina el **bajo sesgo** de las particiones no lineales con la **reducción de varianza** que otorga promediar muchos árboles decorrelacionados (*bagging* y subconjuntos aleatorios de variables). Es invariante a la escala y a la colinealidad del bloque macroeconómico.
2. **K-Nearest Neighbors ($\text{ROC-AUC} \approx 0.773$, $k = 401$, set 3):**
   - *Fundamento:* Método no paramétrico local. Con $k=5$ sufría alta varianza; con vecindarios grandes mejora el ranking, pero su F1 cae a 0.317 porque, con 89% de `no`, el voto casi siempre favorece a la clase mayoritaria.
3. **Support Vector Machines ($\text{ROC-AUC} \approx 0.768$, $C = 10$, set 2):**
   - *Fundamento:* Con solo 7 variables y kernel RBF casi no sobreajusta (brecha 0.011) y logra el segundo mejor F1 (0.453), pero es el modelo más costoso computacionalmente.
4. **Gaussian Naive Bayes ($\text{ROC-AUC} \approx 0.752$, set 2):**
   - *Fundamento:* Modelo de **alto sesgo**: frontera rígida y supuestos violados (independencia y normalidad). Es inmune al sobreajuste, pero no puede capturar interacciones complejas.


In [2]:
resumen_modelos = pd.DataFrame({
    "Modelo": ["Random Forest", "KNN", "SVM", "Naive Bayes"],
    "ROC-AUC (Val)": [0.798, 0.773, 0.768, 0.752],
    "F1-score (Val)": [0.465, 0.317, 0.453, 0.343],
    "Generalization Gap": [0.028, 0.011, 0.011, 0.000],
    "Diagnóstico Sesgo-Varianza": [
        "Bajo sesgo + varianza controlada (profundidad limitada)",
        "Varianza controlada con k grande, sesgo hacia la clase mayoritaria",
        "Sesgo medio, casi sin sobreajuste",
        "Alto sesgo (frontera rígida paramétrica)"
    ]
}).set_index("Modelo")
resumen_modelos

,ROC-AUC (Val),F1-score (Val),Generalization Gap,Diagnóstico Sesgo-Varianza
Modelo,,,,
Random Forest,0.798,0.465,0.028,Bajo sesgo + varianza controlada (profundidad ...
KNN,0.773,0.317,0.011,"Varianza controlada con k grande, sesgo hacia ..."
SVM,0.768,0.453,0.011,"Sesgo medio, casi sin sobreajuste"
Naive Bayes,0.752,0.343,0.000,Alto sesgo (frontera rígida paramétrica)


## 5.2 ¿Los datos satisfacen los supuestos de cada modelo?

Examinamos cómo las propiedades empíricas del dataset desafían los supuestos teóricos de cada familia algorítmica:
- **Independencia condicional:** Violada por Naive Bayes debido a la colinealidad macroeconómica. Random Forest y SVM son robustos.
- **Normalidad gaussiana:** Violada por distribuciones asimétricas y variables binarias en Naive Bayes.
- **Linealidad:** Las relaciones con la variable objetivo no son lineales (tasas muy distintas por mes y por nivel de los indicadores económicos); por eso se usa SVM con kernel RBF y modelos basados en árboles.
- **Escala de variables:** SVM y KNN exigen estandarización estricta (`StandardScaler`). Random Forest es estrictamente invariante a la escala.
- **Desbalance de clases ($11\%$ yes):** Requiere compensación de pesos (`balanced`) o ajuste de umbrales en todos los modelos.

In [3]:
supuestos = pd.DataFrame({
    "Supuesto Teórico": [
        "Independencia condicional entre predictoras",
        "Distribución Normal (Gaussiana) de las variables",
        "Separabilidad lineal de las clases",
        "Invarianza a la escala de las variables",
        "Balance simétrico de clases"
    ],
    "Evidencia en los Datos": [
        "Variables macroeconómicas correlacionadas",
        "Distribuciones asimétricas y variables 0/1",
        "Relaciones no lineales con y",
        "Variables continuas con rangos dispares",
        "Desbalance 88.7% 'no' vs 11.3% 'yes'"
    ],
    "Naive Bayes": ["✘ Violado", "✘ Violado", "N/A", "✔ Cumple", "⚠️ Sensible"],
    "SVM": ["✔ Robusto", "✔ Robusto", "⚠️ Requiere kernel RBF", "⚠️ Requiere Scaler", "⚠️ Sensible"],
    "KNN": ["⚠️ Distorsiona distancias", "✔ Robusto", "✔ Flexible", "⚠️ Requiere Scaler", "⚠️ Sensible"],
    "Random Forest": ["✔ Robusto", "✔ Robusto", "✔ Flexible", "✔ Robusto", "⚠️ Requiere balanced"]
}).set_index("Supuesto Teórico")
supuestos

,Evidencia en los Datos,Naive Bayes,SVM,KNN,Random Forest
Supuesto Teórico,,,,,
Independencia condicional entre predictoras,Variables macroeconómicas correlacionadas,✘ Violado,✔ Robusto,⚠️ Distorsiona distancias,✔ Robusto
Distribución Normal (Gaussiana) de las variables,Distribuciones asimétricas y variables 0/1,✘ Violado,✔ Robusto,✔ Robusto,✔ Robusto
Separabilidad lineal de las clases,Relaciones no lineales con y,N/A,⚠️ Requiere kernel RBF,✔ Flexible,✔ Flexible
Invarianza a la escala de las variables,Variables continuas con rangos dispares,✔ Cumple,⚠️ Requiere Scaler,⚠️ Requiere Scaler,✔ Robusto
Balance simétrico de clases,Desbalance 88.7% 'no' vs 11.3% 'yes',⚠️ Sensible,⚠️ Sensible,⚠️ Sensible,⚠️ Requiere balanced


## 5.3 ¿Qué clasificación de errores es más relevante para el problema de negocio?

### Asimetría Crítica de Costos:
- **Falso Negativo (FN):** Un cliente dispuesto a contratar un depósito a plazo al que el banco no contacta. Representa una pérdida irreparable de captación de pasivos y rentabilidad financiera (**Error Crítico**).
- **Falso Positivo (FP):** Una llamada telefónica a un cliente que no suscribe. Solo representa el costo operativo marginal de la llamada telefónica (**Error Aceptable**).

Por ello, priorizar métricas como **ROC-AUC y Recall** es significativamente superior a maximizar Accuracy.

## 5.4 Limitaciones del Sistema y Mejoras Propuestas

### Limitaciones Prácticas:
1. **Dependencia Temporal del Ciclo Económico:** Datos del periodo 2008–2010 (crisis financiera europea). El Euribor y el índice de confianza dominan la decisión; en un ciclo económico diferente, el modelo requerirá recalibración.
2. **Exclusión de la variable `duration`:** Indispensable para evitar *data leakage* previo a la llamada, pero impone un techo intrínseco al problema ($\text{ROC-AUC} \approx 0.80$).
3. **Falta de Variables Transaccionales del Cliente:** Ausencia de datos bancarios internos (saldo de cuentas corrientes, tenencia de hipotecas, productos de inversión).

### Mejoras Propuestas:
1. **Optimización con Matriz de Costos Reales:** Definir el umbral maximizando la función de utilidad monetaria: $\text{Utilidad} = \text{TP} \cdot \text{Beneficio}_{\text{depósito}} - (\text{TP} + \text{FP}) \cdot \text{Costo}_{\text{llamada}}$.
2. **Validación Temporal (*Time-Series Split*):** Entrenar con los primeros meses y evaluar en meses futuros para simular de forma realista la implementación en producción.
3. **Ensamble Híbrido (*Blending*):** Promediar las probabilidades de Random Forest (reglas globales) con KNN (densidad local) para suavizar decisiones en casos dudosos.

## Mensaje Final de Cierre
El clasificador **Random Forest (`max_depth = 8`, `class_weight = "balanced"`)** operando sobre el set 1 (`max_depth = 8`) representa la solución óptima del proyecto: obtiene la mayor capacidad predictiva ($\text{ROC-AUC} = 0.798$ en validación cruzada y $0.813$ en test), es robusto a las violaciones de supuestos y tiene inferencia rápida.